# 03 - Extracción estructurada de datos desde reportes ECG PDF

Este notebook corresponde al tercer proceso técnico del TFM. Su objetivo es transformar reportes de electrocardiograma en PDF en un dataset tabular estructurado, apto para integración posterior con la cohorte clínica anonimizada.

El proceso genera una clave de emparejamiento compatible con el `crosswalk_paciente_ecg.xlsx` producido por el notebook 01. La integración clínica-ECG no se realiza aquí; queda reservada para el notebook 04.

## 1. Entradas y salidas

### Entrada principal

```text
Carpeta raíz con reportes ECG en PDF
```

### Salidas principales

```text
ecg_dataset.csv
ecg_dataset.xlsx
ecg_resumen.txt
```

### Campos críticos para integración posterior

```text
ECG_ID
archivo_origen
nombre_paciente_norm
fecha
clave_matching
num_ecg_mismo_paciente_fecha
duplicado_mismo_dia
```

`clave_matching` debe ser compatible con la clave generada en el notebook 01 revisado:

```text
NOMBRE_PACIENTE_NORMALIZADO | YYYY-MM-DD
```

In [1]:
# Instalación de dependencias requeridas
# Ejecutar esta celda si el entorno no tiene las dependencias instaladas.

import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "xlsxwriter": "XlsxWriter",
    "pypdfium2": "pypdfium2",
}

for modulo, paquete in DEPENDENCIAS.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando dependencia faltante: {paquete}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
    else:
        print(f"Dependencia disponible: {modulo}")


Dependencia disponible: pandas
Dependencia disponible: openpyxl
Dependencia disponible: xlsxwriter
Dependencia disponible: pypdfium2


In [2]:
# Librerías

from pathlib import Path
from datetime import datetime
import glob
import hashlib
import os
import re
import sys
import traceback
import unicodedata

import pandas as pd
import pypdfium2 as pdfium


## 2. Configuración de rutas

Modificar `ROOT` y `OUT_DIR` según la ubicación local de los PDFs ECG y la carpeta de salida.

En ejecución local Windows, usar rutas absolutas. Ejemplo:

```python
ROOT = Path(r"C:/ECG/ELECTROCARDIOGRAMA")
OUT_DIR = Path(r"C:/ECG")
```

In [3]:
# Configuración de rutas

ROOT = Path(r"C:/ECG/ELECTROCARDIOGRAMA")
OUT_DIR = Path(r"C:/ECG")

# Para prueba dentro del entorno del notebook, se puede usar:
# ROOT = Path(".")
# OUT_DIR = Path(".")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


ROOT: C:\ECG\ELECTROCARDIOGRAMA
OUT_DIR: C:\ECG


## 3. Constantes y funciones de normalización

Las funciones de normalización deben mantenerse alineadas con el notebook 01 revisado para evitar diferencias artificiales en `clave_matching`.

In [4]:
# Constantes

MESES_ES = {
    "ENERO": "01", "FEBRERO": "02", "MARZO": "03", "ABRIL": "04",
    "MAYO": "05", "JUNIO": "06", "JULIO": "07", "AGOSTO": "08",
    "SEPTIEMBRE": "09", "OCTUBRE": "10", "NOVIEMBRE": "11", "DICIEMBRE": "12",
}

NUM = r"(-?\d[\d ,.]*\d|-?\d)"

CORE_ECG_COLS = ["ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC", "ECG_AXIS"]


In [5]:
def to_num(val):
    """Convierte valores numéricos extraídos desde texto PDF a int/float o None."""
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return val

    s = str(val).strip()
    if s == "" or s.upper() == "NULL":
        return None

    s = s.replace(" ", "")
    s = s.replace(",", ".")

    try:
        f = float(s)
        return int(f) if f.is_integer() else f
    except Exception:
        return None


def normalize_name(name):
    """Normaliza nombres para emparejamiento determinístico con notebook 01.

    Reglas:
    - elimina extensión PDF;
    - elimina sufijos duplicados finales como ' 2', '_2', '-2';
    - elimina acentos;
    - convierte a mayúsculas;
    - colapsa espacios múltiples.
    """
    if name is None or pd.isna(name):
        return None

    s = os.path.basename(str(name))
    s = re.sub(r"\.pdf$", "", s, flags=re.I)
    s = re.sub(r"[\s_\-]+(\d+)\s*$", "", s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[^A-Za-zÑñ\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip().upper()
    return s or None


def parse_fecha_segura(value):
    """Parsea fechas evitando advertencias por dayfirst=True en fechas ISO."""
    if value is None or pd.isna(value):
        return pd.NaT

    s = str(value).strip()
    if not s:
        return pd.NaT

    if re.match(r"^\d{4}-\d{2}-\d{2}", s):
        return pd.to_datetime(s, errors="coerce", dayfirst=False)

    return pd.to_datetime(s, errors="coerce", dayfirst=True)


def fecha_to_iso(fecha_examen):
    """Convierte fecha extraída del ECG a YYYY-MM-DD."""
    dt = parse_fecha_segura(fecha_examen)
    if pd.isna(dt):
        return None
    return dt.strftime("%Y-%m-%d")


def build_clave_matching(nombre_norm, fecha_iso):
    """Construye clave de integración compatible con el crosswalk del notebook 01."""
    if not nombre_norm or not fecha_iso:
        return None
    return f"{nombre_norm} | {fecha_iso}"


def build_hash(value, salt="TFM_ECG_PRIVATE_TRACE"):
    """Genera hash auxiliar reproducible. No reemplaza la clave determinística."""
    if not value:
        return None
    raw = f"{value}|{salt}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


## 4. Funciones de extracción desde PDF

In [6]:
def search(pat, text, flags=re.IGNORECASE):
    m = re.search(pat, text, flags)
    return m.group(1).strip() if m else None


def search2(pat, text, flags=re.IGNORECASE):
    m = re.search(pat, text, flags)
    if not m:
        return None, None
    return m.group(1).strip(), m.group(2).strip()


def extract_text(pdf_path):
    """Extrae texto consolidado desde todas las páginas del PDF."""
    doc = pdfium.PdfDocument(str(pdf_path))
    parts = []

    for i in range(len(doc)):
        page = doc[i]
        tp = page.get_textpage()
        txt = tp.get_text_range()
        parts.append(txt)
        tp.close()
        page.close()

    doc.close()

    raw = "\n".join(parts)
    norm = re.sub(r"[\r\n]+", " ", raw)
    norm = re.sub(r"\s+", " ", norm).strip()
    return raw, norm

In [7]:
def is_noise(line):
    """Detecta líneas no clínicas: trazados, parámetros sueltos, cabeceras y metadatos."""
    s = str(line).strip()
    if not s:
        return True

    if re.fullmatch(r"[\d\s,./]+", s) and re.search(r"\d", s) and len(re.findall(r"\d", s)) > len(s) * 0.5:
        return True

    if re.fullmatch(r"((I{1,3}|aVR|aVL|aVF|V[1-6])\s*)+", s):
        return True

    if re.match(r"^(HR|PR|QRS|QT|QTc|RV\d|SV\d|QRS Axis|Name|Pati|ECG No|Age|Sex)", s, re.I):
        return True

    if "ECG Report" in s or "Rpt.Date" in s or "Exam/Dr" in s:
        return True

    if re.match(r"\d{2}-\d{2}-\d{4}", s):
        return True

    if "10mm/mV" in s or "25mm/s" in s:
        return True

    if "ECG Parameters" in s or "ECG Analysis" in s or "ECG Diagnosis" in s:
        return True

    if re.fullmatch(r"(bpm|bmp|ms|mV|Deg\.?)", s, re.I):
        return True

    if re.fullmatch(r"=?\s*-?\d[\d ,.]*\s*(ms|mV|Deg\.?|bpm|bmp)?", s, re.I):
        return True

    return False


def looks_like_clinical_prose(line):
    """Identifica texto clínico interpretable en Analysis/Diagnosis."""
    s = str(line).strip()
    if not s or is_noise(s):
        return False

    letters = re.findall(r"[A-Za-zÁÉÍÓÚáéíóúÑñ]{2,}", s)
    if len(letters) < 2:
        return False

    long_words = [w for w in letters if len(w) >= 4]
    noise_words = {"Undefined", "HospitalECG"}
    long_words = [w for w in long_words if w not in noise_words]
    return len(long_words) >= 1


def extract_analysis_diagnosis(text):
    """Extrae prosa clínica potencial desde el bloque Analysis/Diagnosis."""
    lines = [l.rstrip() for l in text.splitlines()]
    textual = []

    for l in lines:
        s = l.strip()
        if not s:
            continue
        if not looks_like_clinical_prose(s):
            continue
        textual.append(s)

    blob = " | ".join(textual).strip()
    if not blob:
        return None, None

    return None, blob


## 5. Parsing estructurado de un ECG PDF

In [8]:
def init_record():
    return {
        "ECG_ID": None,
        "archivo_origen": None,
        "ruta_relativa": None,
        "año": None,
        "mes": None,
        "fecha_examen": None,
        "paciente_id_pdf": None,
        "sexo": None,
        "edad": None,
        "ECG_HR": None,
        "ECG_PR": None,
        "ECG_QRS": None,
        "ECG_QTC": None,
        "ECG_AXIS": None,
        "QT": None,
        "QRS_AXIS": None,
        "RV5": None,
        "SV1": None,
        "RV1": None,
        "SV5": None,
        "ECG_ANALYSIS": None,
        "ECG_DIAGNOSIS": None,
        "pdf_valido": 0,
        "parametros_extraidos": 0,
        "observaciones_extraccion": "",
        "nombre_paciente": None,
        "nombre_paciente_norm": None,
        "fecha": None,
        "clave_matching": None,
        "clave_hash_ecg": None,
        "num_ecg_mismo_paciente_fecha": 0,
        "duplicado_mismo_dia": 0,
        "estado_match_ecg": "NO_EVALUADO",
    }


In [9]:
def parse_ecg(pdf_path, root=ROOT):
    """Extrae variables estructuradas desde un reporte ECG PDF."""
    pdf_path = Path(pdf_path)
    root = Path(root)
    rec = init_record()
    obs = []

    try:
        try:
            rel = pdf_path.relative_to(root)
        except ValueError:
            rel = Path(os.path.relpath(str(pdf_path), str(root)))

        rec["ruta_relativa"] = str(rel).replace("\\", "/")
        rec["archivo_origen"] = pdf_path.name

        parts = [p for p in rec["ruta_relativa"].split("/") if p]
        year_idx = None
        for idx, p in enumerate(parts):
            if re.fullmatch(r"\d{4}", p) and int(p) >= 1900:
                rec["año"] = p
                year_idx = idx
                break

        if year_idx is not None:
            for p in parts[year_idx + 1:]:
                up = p.upper()
                if up in MESES_ES:
                    rec["mes"] = MESES_ES[up]
                    break
                if re.fullmatch(r"\d{2}", p) and 1 <= int(p) <= 12:
                    rec["mes"] = p
                    break

        text, norm = extract_text(pdf_path)
        if not text or not text.strip():
            rec["pdf_valido"] = 0
            rec["observaciones_extraccion"] = "PDF sin texto extraible"
            return rec

        rec["pdf_valido"] = 1

        m = re.search(r"Pati\.?ID\s*(\S+)", norm, re.I)
        if m:
            rec["paciente_id_pdf"] = m.group(1).strip()

        m = re.search(r"Name\s+(.*?)\s*Sex\s*([MFmf])\s*Age\s*(\d+)\s*Y", norm, re.I | re.S)
        if m:
            rec["sexo"] = m.group(2).upper()
            rec["edad"] = to_num(m.group(3))
        else:
            ms = re.search(r"Sex\s*([MFmf])", norm, re.I)
            if ms:
                rec["sexo"] = ms.group(1).upper()
            me = re.search(r"Age\s*(\d+)\s*Y", norm, re.I)
            if me:
                rec["edad"] = to_num(me.group(1))
            if not ms or not me:
                obs.append("bloque Name/Sex/Age no coincide con patron completo")

        mf = re.search(r"(\d{2}-\d{2}-\d{4}\s+\d{1,2}:\d{2}(?::\d{2})?)", norm)
        if mf:
            rec["fecha_examen"] = mf.group(1).strip()

        rec["nombre_paciente"] = pdf_path.stem
        rec["nombre_paciente_norm"] = normalize_name(pdf_path.name)
        rec["fecha"] = fecha_to_iso(rec["fecha_examen"])
        rec["clave_matching"] = build_clave_matching(rec["nombre_paciente_norm"], rec["fecha"])
        rec["clave_hash_ecg"] = build_hash(rec["clave_matching"])

        v = search(r"HR\s*=?\s*" + NUM + r"\s*(?:bpm|bmp|b/min)?", norm)
        rec["ECG_HR"] = to_num(v)

        v = search(r"PR\s*=?\s*" + NUM + r"\s*ms", norm)
        rec["ECG_PR"] = to_num(v)

        v = search(r"QRS\s*=?\s*" + NUM + r"\s*ms", norm)
        rec["ECG_QRS"] = to_num(v)

        v = search(r"QRS\s*Axis\s*=?\s*" + NUM, norm)
        rec["ECG_AXIS"] = to_num(v)
        rec["QRS_AXIS"] = rec["ECG_AXIS"]

        a, b = search2(r"QT\s*/\s*QTc\s*=?\s*" + NUM + r"\s*/\s*" + NUM, norm)
        if a is not None:
            rec["QT"] = to_num(a)
            rec["ECG_QTC"] = to_num(b)
        else:
            v2 = search(r"QTc\s*=?\s*" + NUM, norm)
            rec["ECG_QTC"] = to_num(v2)

        a, b = search2(r"RV5\s*/\s*SV1\s*=?\s*" + NUM + r"\s*/\s*" + NUM, norm)
        if a is not None:
            rec["RV5"] = to_num(a)
            rec["SV1"] = to_num(b)

        a, b = search2(r"RV1\s*/\s*SV5\s*=?\s*" + NUM + r"\s*/\s*" + NUM, norm)
        if a is not None:
            rec["RV1"] = to_num(a)
            rec["SV5"] = to_num(b)

        analysis, diagnosis = extract_analysis_diagnosis(text)
        rec["ECG_ANALYSIS"] = analysis
        rec["ECG_DIAGNOSIS"] = diagnosis

        core = [rec[c] for c in CORE_ECG_COLS]
        rec["parametros_extraidos"] = sum(1 for x in core if x is not None)
        missing = [k for k, v in zip(["HR", "PR", "QRS", "QTc", "Axis"], core) if v is None]
        if missing:
            obs.append("parametros faltantes: " + ",".join(missing))

        if not rec["clave_matching"]:
            rec["estado_match_ecg"] = "SIN_CLAVE_MATCHING"
            obs.append("clave_matching no construida")
        elif missing:
            rec["estado_match_ecg"] = "CLAVE_OK_PARAMETROS_INCOMPLETOS"
        else:
            rec["estado_match_ecg"] = "CLAVE_OK_PARAMETROS_CORE_OK"

        rec["observaciones_extraccion"] = "; ".join(o for o in obs if o)

    except Exception as e:
        rec["pdf_valido"] = 0
        rec["estado_match_ecg"] = "ERROR_EXTRACCION"
        rec["observaciones_extraccion"] = "ERROR: " + repr(e)[:300]

    return rec

## 6. Procesamiento batch de PDFs

In [10]:
def discover_pdfs(root=ROOT):
    root = Path(root)
    return sorted(root.glob("**/*.pdf"))


def build_ecg_dataset(root=ROOT):
    pdfs = discover_pdfs(root)
    print("PDFs detectados:", len(pdfs))

    rows = []
    for i, p in enumerate(pdfs, 1):
        rec = parse_ecg(p, root=root)
        rows.append(rec)
        sys.stdout.write(f"\r[{i}/{len(pdfs)}] {p.name[:60]:60s}")
        sys.stdout.flush()

    if pdfs:
        print()

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df.insert(0, "ECG_ID_TEMP", range(1, len(df) + 1))
    df["ECG_ID"] = df["ECG_ID_TEMP"].apply(lambda x: f"ECG_{x:06d}")
    df = df.drop(columns=["ECG_ID_TEMP"])

    counts = df.groupby("clave_matching")["clave_matching"].transform("count")
    df["num_ecg_mismo_paciente_fecha"] = counts.fillna(0).astype(int)
    df["duplicado_mismo_dia"] = ((df["num_ecg_mismo_paciente_fecha"] > 1) & df["clave_matching"].notna()).astype(int)

    df["rank_ecg_mismo_paciente_fecha"] = (
        df.groupby("clave_matching").cumcount() + 1
    ).where(df["clave_matching"].notna(), 0).astype(int)

    cols = [
        "ECG_ID", "archivo_origen", "ruta_relativa", "año", "mes", "fecha_examen",
        "paciente_id_pdf", "sexo", "edad", "ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC",
        "ECG_AXIS", "QT", "QRS_AXIS", "RV5", "SV1", "RV1", "SV5",
        "ECG_ANALYSIS", "ECG_DIAGNOSIS", "pdf_valido", "parametros_extraidos",
        "observaciones_extraccion", "nombre_paciente", "nombre_paciente_norm",
        "fecha", "clave_matching", "clave_hash_ecg",
        "num_ecg_mismo_paciente_fecha", "duplicado_mismo_dia",
        "rank_ecg_mismo_paciente_fecha", "estado_match_ecg",
    ]

    cols = [c for c in cols if c in df.columns]
    return df[cols]

## 7. Validación y reporte de calidad

In [11]:
def build_resumen(df):
    total = len(df)
    lines = []
    lines.append("RESUMEN EXTRACCION ECG")
    lines.append("=" * 60)
    lines.append(f"Fecha generacion : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"Total PDFs procesados         : {total}")

    if total == 0:
        lines.append("Sin PDFs procesados.")
        return "\n".join(lines)

    validos = int((df["pdf_valido"] == 1).sum())
    errores = int((df["pdf_valido"] == 0).sum())
    completos = int(df[CORE_ECG_COLS].notna().all(axis=1).sum())
    claves_ok = int(df["clave_matching"].notna().sum())
    duplicados = int(df["duplicado_mismo_dia"].sum())

    lines.append(f"Total ECG validos             : {validos}")
    lines.append(f"Total ECG con error           : {errores}")
    lines.append(f"ECG con 5 parametros core OK  : {completos}")
    lines.append(f"ECG con clave_matching        : {claves_ok}")
    lines.append(f"ECG duplicados mismo dia      : {duplicados}")
    lines.append("")

    var_cols = [
        "paciente_id_pdf", "sexo", "edad", "ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC",
        "ECG_AXIS", "QT", "RV5", "SV1", "RV1", "SV5", "ECG_ANALYSIS", "ECG_DIAGNOSIS",
        "nombre_paciente_norm", "fecha", "clave_matching",
    ]

    lines.append("COMPLETITUD POR VARIABLE (%)")
    lines.append("-" * 60)
    for c in var_cols:
        if c in df.columns:
            n = int(df[c].notna().sum())
            pct = round(n / total * 100, 1)
            lines.append(f"{c:30s}: {pct:5.1f}%  ({n}/{total})")
    lines.append("")

    lines.append("DISTRIBUCION POR ANIO")
    lines.append("-" * 60)
    for k, v in df["año"].value_counts(dropna=False).sort_index().items():
        lines.append(f"  {k}: {v}")
    lines.append("")

    lines.append("DISTRIBUCION POR MES")
    lines.append("-" * 60)
    for k, v in df["mes"].value_counts(dropna=False).sort_index().items():
        lines.append(f"  {k}: {v}")
    lines.append("")

    lines.append("DISTRIBUCION DE parametros_extraidos")
    lines.append("-" * 60)
    for n in range(6):
        cnt = int((df["parametros_extraidos"] == n).sum())
        lines.append(f"  {n} parametros: {cnt}")
    lines.append("")

    lines.append("VALIDACION DE CLAVES ECG")
    lines.append("-" * 60)
    lines.append("Estrategia: nombre_paciente_norm + fecha")
    lines.append(f"Claves matching unicas         : {df['clave_matching'].nunique(dropna=True)}")
    lines.append(f"Registros con clave            : {claves_ok}")
    lines.append(f"Registros sin clave            : {total - claves_ok}")
    lines.append(f"Registros duplicados           : {duplicados}")
    lines.append("")

    dup_groups = df[df["duplicado_mismo_dia"] == 1].groupby("clave_matching").size().sort_values(ascending=False)
    lines.append("CLAVES CON MAS DE UN ECG")
    lines.append("-" * 60)
    if len(dup_groups) == 0:
        lines.append("  Sin duplicados.")
    else:
        for k, v in dup_groups.items():
            lines.append(f"  {v}x  {k}")
    lines.append("")

    lines.append("OBSERVACIONES DE EXTRACCION")
    lines.append("-" * 60)
    nz = df[df["observaciones_extraccion"].fillna("").astype(str).str.len() > 0]
    if len(nz) == 0:
        lines.append("  Sin observaciones.")
    else:
        for _, r in nz.iterrows():
            lines.append(f"  - {r['archivo_origen']}: {r['observaciones_extraccion']}")

    lines.append("")
    lines.append("NOTA METODOLOGICA")
    lines.append("-" * 60)
    lines.append("paciente_id_pdf no debe usarse como identificador clínico real.")
    lines.append("La integración posterior debe usar crosswalk_paciente_ecg.xlsx.")

    return "\n".join(lines)

## 8. Exportación de resultados

In [12]:
def export_outputs(df, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / "ecg_dataset.csv"
    xlsx_path = out_dir / "ecg_dataset.xlsx"
    txt_path = out_dir / "ecg_resumen.txt"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name="ECG_DATASET", index=False)

        resumen_df = pd.DataFrame({
            "metrica": [
                "total_registros",
                "pdf_validos",
                "core_ecg_completo",
                "claves_matching_disponibles",
                "duplicados_mismo_paciente_fecha",
            ],
            "valor": [
                len(df),
                int((df["pdf_valido"] == 1).sum()) if len(df) else 0,
                int(df[CORE_ECG_COLS].notna().all(axis=1).sum()) if len(df) else 0,
                int(df["clave_matching"].notna().sum()) if len(df) else 0,
                int(df["duplicado_mismo_dia"].sum()) if len(df) else 0,
            ],
        })
        resumen_df.to_excel(writer, sheet_name="RESUMEN", index=False)

        if len(df):
            duplicados_df = df[df["duplicado_mismo_dia"] == 1].copy()
            duplicados_df.to_excel(writer, sheet_name="DUPLICADOS", index=False)

    resumen = build_resumen(df)
    txt_path.write_text(resumen, encoding="utf-8")

    print(resumen)
    print()
    print("ARCHIVOS GENERADOS:")
    print("CSV :", csv_path)
    print("XLSX:", xlsx_path)
    print("TXT :", txt_path)

    return csv_path, xlsx_path, txt_path


## 9. Ejecución principal

Ejecutar esta celda cuando `ROOT` apunte a la carpeta real de PDFs ECG.

In [13]:
df_ecg = build_ecg_dataset(ROOT)
print("Shape ECG dataset:", df_ecg.shape)
df_ecg.head()


PDFs detectados: 2679
[2679/2679] SEBASTIAN.pdf                                               
Shape ECG dataset: (2679, 34)


,ECG_ID,archivo_origen,ruta_relativa,año,mes,fecha_examen,paciente_id_pdf,sexo,edad,ECG_HR,...,observaciones_extraccion,nombre_paciente,nombre_paciente_norm,fecha,clave_matching,clave_hash_ecg,num_ecg_mismo_paciente_fecha,duplicado_mismo_dia,rank_ecg_mismo_paciente_fecha,estado_match_ecg
0,ECG_000001,BRENDA OCHOA MARTINEZ 2.pdf,2024/ABRIL/01/BRENDA OCHOA MARTINEZ 2.pdf,2024,04,01-04-2024 9:31:50,3,M,37.0,59.0,...,,BRENDA OCHOA MARTINEZ 2,BRENDA OCHOA MARTINEZ,2024-04-01,BRENDA OCHOA MARTINEZ | 2024-04-01,a1c2680594f62f63968ac3ec3212bcea45a17d8bc055bf...,2,1,1,CLAVE_OK_PARAMETROS_CORE_OK
1,ECG_000002,BRENDA OCHOA MARTINEZ.pdf,2024/ABRIL/01/BRENDA OCHOA MARTINEZ.pdf,2024,04,01-04-2024 9:28:45,3,M,37.0,63.0,...,,BRENDA OCHOA MARTINEZ,BRENDA OCHOA MARTINEZ,2024-04-01,BRENDA OCHOA MARTINEZ | 2024-04-01,a1c2680594f62f63968ac3ec3212bcea45a17d8bc055bf...,2,1,2,CLAVE_OK_PARAMETROS_CORE_OK
2,ECG_000003,FELIPE AQUEZ PERALTA.pdf,2024/ABRIL/01/FELIPE AQUEZ PERALTA.pdf,2024,04,01-04-2024 9:52:34,3,M,35.0,73.0,...,,FELIPE AQUEZ PERALTA,FELIPE AQUEZ PERALTA,2024-04-01,FELIPE AQUEZ PERALTA | 2024-04-01,4d2e337db087e62fd28e7974260ba7482962178e8532c3...,1,0,1,CLAVE_OK_PARAMETROS_CORE_OK
3,ECG_000004,JAIME GUZMAN COLLAO.pdf,2024/ABRIL/01/JAIME GUZMAN COLLAO.pdf,2024,04,01-04-2024 11:23:09,3,M,37.0,55.0,...,,JAIME GUZMAN COLLAO,JAIME GUZMAN COLLAO,2024-04-01,JAIME GUZMAN COLLAO | 2024-04-01,e3923bfb9994c56ad5a1ecd142c1f59992440f1d3548e7...,1,0,1,CLAVE_OK_PARAMETROS_CORE_OK
4,ECG_000005,CRISTIAN GUZMAN ASENCIO.pdf,2024/ABRIL/02/CRISTIAN GUZMAN ASENCIO.pdf,2024,04,02-04-2024 8:27:40,3,M,54.0,46.0,...,,CRISTIAN GUZMAN ASENCIO,CRISTIAN GUZMAN ASENCIO,2024-04-02,CRISTIAN GUZMAN ASENCIO | 2024-04-02,56711961622f80a4bbe4d6942ca9c4e532991cd4cb74e7...,1,0,1,CLAVE_OK_PARAMETROS_CORE_OK


In [14]:
# Exportar resultados

csv_path, xlsx_path, txt_path = export_outputs(df_ecg, OUT_DIR)


RESUMEN EXTRACCION ECG
Fecha generacion : 2026-07-22 09:28:33
Total PDFs procesados         : 2679
Total ECG validos             : 2678
Total ECG con error           : 1
ECG con 5 parametros core OK  : 2655
ECG con clave_matching        : 2678
ECG duplicados mismo dia      : 108

COMPLETITUD POR VARIABLE (%)
------------------------------------------------------------
paciente_id_pdf               : 100.0%  (2678/2679)
sexo                          :  99.9%  (2675/2679)
edad                          :  99.4%  (2663/2679)
ECG_HR                        :  99.7%  (2670/2679)
ECG_PR                        :  99.4%  (2662/2679)
ECG_QRS                       :  99.7%  (2670/2679)
ECG_QTC                       :  99.6%  (2667/2679)
ECG_AXIS                      :  99.4%  (2663/2679)
QT                            :  99.6%  (2667/2679)
RV5                           :  54.5%  (1461/2679)
SV1                           :  54.5%  (1461/2679)
RV1                           :  54.5%  (1461/2679)
SV5  

## 10. Prueba opcional con un PDF individual

Usar esta sección para validar la extracción sobre un único archivo PDF antes de procesar la carpeta completa.

In [15]:
# Prueba opcional con un PDF individual
# Modificar SAMPLE_PDF si se requiere validar otro archivo.

SAMPLE_PDF = Path("FELIPE AQUEZ PERALTA.pdf")

if SAMPLE_PDF.exists():
    sample_rec = parse_ecg(SAMPLE_PDF, root=SAMPLE_PDF.parent)
    display(pd.DataFrame([sample_rec]))
else:
    print("PDF de prueba no encontrado en el directorio actual:", SAMPLE_PDF)


PDF de prueba no encontrado en el directorio actual: FELIPE AQUEZ PERALTA.pdf


## 11. Consideraciones de privacidad

`ecg_dataset.xlsx` puede contener nombres de pacientes derivados de los nombres de archivo de los PDFs. Por tanto, no debsera publicado en repositorios públicos.

Para uso público se conservara únicamente:

```text
notebook
script de extracción
diccionario de variables
salidas sintéticas o anonimizadas
```

No se publicara:

```text
PDFs ECG originales
ecg_dataset.xlsx real
ecg_dataset.csv real
crosswalk_paciente_ecg.xlsx
archivos con nombres o claves derivadas de nombres
```

## 12. Relación con notebooks posteriores

Este notebook no integra la base clínica con los ECG. Solo produce el dataset ECG estructurado.

La integración determinística debe ejecutarse en el notebook 04 usando:

```text
Base_Ordenada_Subsets_TFM.xlsx
crosswalk_paciente_ecg.xlsx
ecg_dataset.xlsx
```

La relación mínima esperada es:

```text
Base clínica: PACIENTE_ID
Crosswalk: PACIENTE_ID + clave_matching
ECG dataset: clave_matching
```